# ChemSim — Quickstart Notebook

This notebook demonstrates the ChemSim Python API.
Make sure the `chemsim` module is on your `PYTHONPATH` or installed via `pip install -e .`.

```
# Build the Python module first (from repo root):
cmake -B build -DCMAKE_BUILD_TYPE=Release
cmake --build build --target chemsim -j
# Then add build/ to PYTHONPATH:
# $env:PYTHONPATH = "C:/path/to/build"
```

In [ ]:
import sys, pathlib
# Adjust this to your actual build directory
build_dir = pathlib.Path("../build")
if str(build_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(build_dir.resolve()))

import chemsim
print("chemsim loaded:", chemsim.__doc__[:40])

## 1 — Single Flash Drum (TP spec)

Separate a methane/ethane/propane feed at 250 K, 2 MPa.

In [ ]:
DB = "../data/components.json"

fs = chemsim.Flowsheet.create(["METHANE", "ETHANE", "PROPANE"], db_path=DB)

# Feed stream
fs.add_stream("FEED", T=250.0, P=2e6, flow=100.0,
              z={"METHANE": 0.5, "ETHANE": 0.3, "PROPANE": 0.2})

# Flash drum at 250 K, 2 MPa
fs.add_flash_drum("FLASH1", T=250.0, P=2e6)

# Connect FEED into FLASH1's 'feed' port
fs.connect("FEED",  to_unit="FLASH1", to_port="feed")
# Declare product streams (from_unit only)
fs.connect("VAPOR", from_unit="FLASH1", from_port="vapor")
fs.connect("LIQ",   from_unit="FLASH1", from_port="liquid")

fs.solve()
print(fs.summary())

In [ ]:
import pandas as pd

df = pd.DataFrame(fs.results_table())
df = df.set_index("name")
df[["T", "P", "total_flow", "vapor_fraction", "H", "phase"]]

In [ ]:
# Inspect a single stream
vapor = fs.get_stream("VAPOR")
print(vapor)
print(f"  Vapor fraction (β):  {vapor.vapor_fraction:.4f}")
print(f"  CH4 in vapor:        {vapor.y[0]:.4f}")
print(f"  C3H8 in liquid:      {fs.get_stream('LIQ').x[2]:.4f}")

## 2 — Flash + Recycle Loop

A simple reflux-like loop:
```
FEED ──► [MIXER] ──► [FLASH] ──► VAPOR (product)
                        │
                        └──► LIQ ──► [SPLIT] ──► LIQ_PROD (70 %)
                                          └──► RECYCLE (30 %) ──► [MIXER]
```

In [ ]:
fs2 = chemsim.Flowsheet.create(["METHANE", "ETHANE"], db_path=DB)

# Feed
fs2.add_stream("FEED", T=250.0, P=2e6, flow=100.0,
               z={"METHANE": 0.6, "ETHANE": 0.4})

# Units
fs2.add_mixer   ("MIX1",  inlet_ports=["feed", "recycle"])
fs2.add_flash_drum("FLASH1", T=250.0, P=2e6)
fs2.add_splitter("SPLIT1", fractions=[0.7, 0.3])  # 70% product, 30% recycle

# Connections
fs2.connect("FEED",     to_unit="MIX1",  to_port="feed")
fs2.connect("MIXED",    from_unit="MIX1",   from_port="out",
                         to_unit="FLASH1",  to_port="feed")
fs2.connect("VAPOR",    from_unit="FLASH1", from_port="vapor")
fs2.connect("LIQ",      from_unit="FLASH1", from_port="liquid",
                         to_unit="SPLIT1",  to_port="in")
fs2.connect("LIQ_PROD", from_unit="SPLIT1", from_port="out0")
fs2.connect("RECYCLE",  from_unit="SPLIT1", from_port="out1",
                         to_unit="MIX1",    to_port="recycle")

converged = fs2.solve_with(max_iter=50, tol_z=1e-6)
print(f"Converged: {converged}")
print(fs2.summary())

In [ ]:
df2 = pd.DataFrame(fs2.results_table()).set_index("name")
df2[["T", "P", "total_flow", "vapor_fraction", "phase"]]

In [ ]:
# Verify overall material balance
F_in  = fs2.get_stream("FEED").total_flow
F_vap = fs2.get_stream("VAPOR").total_flow
F_liq = fs2.get_stream("LIQ_PROD").total_flow
print(f"Feed:     {F_in:.2f} mol/s")
print(f"Vapor:    {F_vap:.2f} mol/s")
print(f"Liq prod: {F_liq:.2f} mol/s")
print(f"Balance error: {abs(F_in - F_vap - F_liq):.2e} mol/s")

## 3 — From JSON

Flowsheets can also be loaded from a JSON specification file.

In [ ]:
# fs3 = chemsim.Flowsheet.from_json(
#     "../examples/simple_recycle.json",
#     "../data/components.json")
# fs3.solve()
# print(fs3.summary())
print("(Uncomment once you have a JSON flowsheet file)")

## 4 — Export results

In [ ]:
import json
data = json.loads(fs2.results_as_json())
print("Streams in result:", list(data["streams"].keys()))
# fs2.export_results("results.json")  # write to file